# Clean a CSV with Pandas


# Author: Mohamed BEN ALI
# Date & time: (November) 1/11/2026 // 12:40


## Project Requirements
- Load the dataset and inspect it before doing anything

- Handle encoding and delimiter issues at load time

- Fix column data types explicitly using the dtype argument

- Convert date columns using pd.to_datetime() with errors='coerce'

- Standardize categorical columns (strip whitespace, fix capitalisation)

- Export a cleaned version of the dataset and do a final audit

## Step 1 — Load the CSV (And Handle the First Hidden Issues)


1. import library


In [1531]:
import pandas as pd

2. import datasets

In [1532]:
df = pd.read_csv("Messy_Employee_dataset.csv")
df

,Employee_ID,First_Name,Last_Name,Age,Department_Region,Status,Join_Date,Salary,Email,Phone,Performance_Score,Remote_Work
0,EMP1000,Bob,Davis,25.0,DevOps-California,Active,4/2/2021,59767.65,bob.davis@example.com,-1651623197,Average,True
1,EMP1001,Bob,Brown,NaN,Finance-Texas,Active,7/10/2020,65304.66,bob.brown@example.com,-1898471390,Excellent,True
2,EMP1002,Alice,Jones,NaN,Admin-Nevada,Pending,12/7/2023,88145.90,alice.jones@example.com,-5596363211,Good,True
3,EMP1003,Eva,Davis,25.0,Admin-Nevada,Inactive,11/27/2021,69450.99,eva.davis@example.com,-3476490784,Good,True
4,EMP1004,Frank,Williams,25.0,Cloud Tech-Florida,Active,1/5/2022,109324.61,frank.williams@example.com,-1586734256,Poor,False
...,...,...,...,...,...,...,...,...,...,...,...,...
1015,EMP2015,David,Miller,30.0,HR-California,Active,8/19/2023,NaN,david.miller@example.com,-3546212759,Good,True
1016,EMP2016,David,Johnson,30.0,Cloud Tech-Texas,Inactive,11/7/2021,100215.06,david.johnson@example.com,-2508261122,Good,True
1017,EMP2017,Charlie,Williams,40.0,Finance-New York,Active,10/4/2023,114587.11,charlie.williams@example.com,-1261632487,Average,False
1018,EMP2018,Alice,Garcia,30.0,HR-Florida,Inactive,12/16/2024,71318.79,alice.garcia@example.com,-8995729892,Good,True


## Step 2 — Inspect the Dataset


In [1533]:
df.shape

(1020, 12)

The datasets have dimensions of 1020 rows and 12 columns.

1. Understanding the Boundaries:




I always start with a visual check. I use **df.head()** and **df.tail()** to see the first and last five rows. This is my quick sanity check to see if all columns look aligned and if the data visually makes sense.

In [1534]:
df.head()

,Employee_ID,First_Name,Last_Name,Age,Department_Region,Status,Join_Date,Salary,Email,Phone,Performance_Score,Remote_Work
0,EMP1000,Bob,Davis,25.0,DevOps-California,Active,4/2/2021,59767.65,bob.davis@example.com,-1651623197,Average,True
1,EMP1001,Bob,Brown,NaN,Finance-Texas,Active,7/10/2020,65304.66,bob.brown@example.com,-1898471390,Excellent,True
2,EMP1002,Alice,Jones,NaN,Admin-Nevada,Pending,12/7/2023,88145.90,alice.jones@example.com,-5596363211,Good,True
3,EMP1003,Eva,Davis,25.0,Admin-Nevada,Inactive,11/27/2021,69450.99,eva.davis@example.com,-3476490784,Good,True
4,EMP1004,Frank,Williams,25.0,Cloud Tech-Florida,Active,1/5/2022,109324.61,frank.williams@example.com,-1586734256,Poor,False


In [1535]:
df.tail()

,Employee_ID,First_Name,Last_Name,Age,Department_Region,Status,Join_Date,Salary,Email,Phone,Performance_Score,Remote_Work
1015,EMP2015,David,Miller,30.0,HR-California,Active,8/19/2023,NaN,david.miller@example.com,-3546212759,Good,True
1016,EMP2016,David,Johnson,30.0,Cloud Tech-Texas,Inactive,11/7/2021,100215.06,david.johnson@example.com,-2508261122,Good,True
1017,EMP2017,Charlie,Williams,40.0,Finance-New York,Active,10/4/2023,114587.11,charlie.williams@example.com,-1261632487,Average,False
1018,EMP2018,Alice,Garcia,30.0,HR-Florida,Inactive,12/16/2024,71318.79,alice.garcia@example.com,-8995729892,Good,True
1019,EMP2019,Heidi,Jones,30.0,DevOps-Illinois,Pending,2/22/2021,77764.24,heidi.jones@example.com,-7629745492,Average,False


My Finding:

- When I ran df.head(), I noticed my **Employee ID** was sitting in a column, and the DataFrame was using the default numerical index (0, 1, 2, ...) instead.

- While I know I could set **Employee ID** as the index, for now, I’ll leave it. The bigger immediate visual risk I’m looking for here is data misaligned in the wrong column or obvious leading/trailing spaces on names that will cause trouble later.

2. Spotting Datatype Problems and Missingness

the column names, the data types (Dtype), and the exact number of non-null values.


In [1536]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Employee_ID        1020 non-null   object 
 1   First_Name         1020 non-null   object 
 2   Last_Name          1020 non-null   object 
 3   Age                809 non-null    float64
 4   Department_Region  1020 non-null   object 
 5   Status             1020 non-null   object 
 6   Join_Date          1020 non-null   object 
 7   Salary             996 non-null    float64
 8   Email              1020 non-null   object 
 9   Phone              1020 non-null   int64  
 10  Performance_Score  1020 non-null   object 
 11  Remote_Work        1020 non-null   bool   
dtypes: bool(1), float64(2), int64(1), object(8)
memory usage: 88.8+ KB


My Findings on 1,020 Rows:

1. **Missing Age**: My total entry count is *1,020*, but the Age column only has *809* non-null values. That’s a significant amount of missing data that I'll have to decide how to handle later—do I impute it, or do I drop the rows?

2. **Missing Salary**: The Salary column has *996* non-null values, which is only a minor gap, but still something I must resolve.

3. **The ID** Type Check: *The Employee ID* is correctly listed as an object (string). This isn't right. IDs are identifiers, not numbers to be averaged, and using the string type prevents Pandas from accidentally stripping leading zeros.

3. Data Integrity Check: Duplicates and Unique Counts


- Checking for Duplicates:

In [1537]:
df.duplicated().sum()

np.int64(0)

There are no duplicate rows.

- Checking Unique Values:

**df.nunique()**: I use this to understand the diversity within each column. Low counts in categorical columns are fine, but I look for columns that should be unique but aren't, or columns that have too many unique values, suggesting typos

In [1538]:
df.nunique()

Employee_ID          1020
First_Name              8
Last_Name               8
Age                     4
Department_Region      36
Status                  3
Join_Date             760
Salary                978
Email                  64
Phone                1020
Performance_Score       4
Remote_Work             2
dtype: int64

My Findings on 1,020 Rows:
- Employee_ID have 1020 unique records. This is perfect. It means all records are unique.
- Phone (1020 unique records). This is perfect because it confirms phone numbers are unique identifiers.
- The First_Name / Last_Name field has eight unique records. That's a little odd. This confirms the dataset's synthetic nature. My analysis won't be skewed by a large variety of names, since I'll treat them as standard strings.
- Department_Region has 36 unique records. There’s high potential for typos. 36 unique values for region/department is too many. I will need to check this column for spelling variations (e.g., “HR” vs. “Human Resources”) in the next step.
- Email (64 unique records). With 1,020 employees, having only 64 unique emails suggests many employees share the same placeholder email. I will flag this for exclusion from analysis, as it’s useless for identifying individuals.
- Age / Performance Score / Status / Remote Work (2–4 unique records). These low counts are expected for categorical or ordinal data, meaning they are ready for encoding.

4. Catching Odd and Impossible Values

I use **df.describe()** to get a statistical summary of all my numerical columns. This is the place where truly impossible values—the "red flags"—show up instantly. I mostly focus on the min and max rows.

In [1539]:
df.describe()

,Age,Salary,Phone
count,809.000000,996.000000,1.020000e+03
mean,32.484549,85155.056396,-4.942253e+09
std,5.656860,19873.727918,2.817326e+09
min,25.000000,50047.320000,-9.994973e+09
25%,25.000000,68392.487500,-7.341992e+09
50%,30.000000,85547.870000,-4.943997e+09
75%,40.000000,100974.027500,-2.520391e+09
max,40.000000,119971.650000,-3.896086e+06


My Findings:

- I immediately noticed a problem in what I expected to be the Phone Number column, which Pandas mistakenly converted to a numerical type.
It turns out all the phone number values were massive negative numbers! This confirms two things:

- Pandas incorrectly inferred this column as a number, even though phone numbers are strings.

- There must be characters in the text that Pandas cannot interpret (for example, parentheses, dashes, or country codes). I need to convert this to an object type and clean it up completely.

5. Quantifying Missing Data

While df.info() gives me non-null counts, df.isnull().sum() gives me the total count of nulls, which is a cleaner way to quantify my next steps.

In [1540]:
df.isnull().sum()

Employee_ID            0
First_Name             0
Last_Name              0
Age                  211
Department_Region      0
Status                 0
Join_Date              0
Salary                24
Email                  0
Phone                  0
Performance_Score      0
Remote_Work            0
dtype: int64

My Findings:

- Age has 211 nulls (1020 - 809 = 211), and

- Salary has 24 nulls (1020 - 996 = 24). This precise count sets the stage for Step 3.

- This inspection process is my safety net. If I had missed the negative phone numbers, any analytical step that involved numerical data would have failed or, worse, produced skewed results without warning.

- By identifying the need to treat Phone Number as a string and the significant missing values in Age now, I have a concrete cleaning list. This prevents runtime errors and, critically, ensures that my final analysis is based on plausible, non-corrupted data.

## Step 3 — Standardise Column Names, Correct Dtypes, and Handle Missing Values

1. Standardising Column Names and Setting the Index (The Consistency Rule)

 Before I do any serious data manipulation, I enforce strict consistency on column names. Why? Because typing df['Employee ID '] accidentally instead of df['employee_id'] is a silent, frustrating error. Once the names are clean, I set the index.

- My golden rule is snake_case and lowercase everywhere, and ID columns should be the index.



Columns before:

In [1541]:
df.columns

Index(['Employee_ID', 'First_Name', 'Last_Name', 'Age', 'Department_Region',
       'Status', 'Join_Date', 'Salary', 'Email', 'Phone', 'Performance_Score',
       'Remote_Work'],
      dtype='object')

I use a simple command to strip whitespace, replace spaces with underscores, and convert everything to lowercase.

In [1542]:
df.columns = df.columns.str.lower().str.replace(' ', '_').str.strip()
# Before: ['Employee_ID', 'First_Name', 'Phone']
# After: ['employee_id', 'first_name', 'phone']

Columns after:

In [1543]:
df.columns 

Index(['employee_id', 'first_name', 'last_name', 'age', 'department_region',
       'status', 'join_date', 'salary', 'email', 'phone', 'performance_score',
       'remote_work'],
      dtype='object')

Index before:

In [1544]:
df.index

RangeIndex(start=0, stop=1020, step=1)

In [1545]:
# Set the Employee ID as the DataFrame Index
# This is crucial for efficient lookups and clean merges later.
df.set_index('employee_id', inplace=True)


Index after:

In [1546]:
print(df.index)


Index(['EMP1000', 'EMP1001', 'EMP1002', 'EMP1003', 'EMP1004', 'EMP1005',
       'EMP1006', 'EMP1007', 'EMP1008', 'EMP1009',
       ...
       'EMP2010', 'EMP2011', 'EMP2012', 'EMP2013', 'EMP2014', 'EMP2015',
       'EMP2016', 'EMP2017', 'EMP2018', 'EMP2019'],
      dtype='object', name='employee_id', length=1020)


2. Fixing Data Types and Corruption (Tackling the Negative Phone Numbers)

My df.describe() check revealed the most urgent structural flaw: the Phone column, which was imported as a garbage numerical type. Since phone numbers are identifiers (not quantities), they must be strings.


In this phase, I'll convert the entire column to a string type, which will turn all those negative scientific notation numbers into human-readable text (though still full of non-digit characters). I will leave the actual text cleaning (removing parentheses, dashes, etc.) for a dedicated standardisation step (Step 4).

In [1547]:
# Fix the Phone dtype immediately
# Note: The column name is now 'phone' due to standardization in 3.1
df['phone'] = df['phone'].astype(str)

3. Handling Missing Values (The Age & Salary Gaps)


Finally, I address the gaps revealed by df.info(): the 211 missing Age values and the 24 missing Salary values (out of 1,020 total rows). My strategy depends entirely on the column's role and the magnitude of the missing data:

- Salary (24 missing values): In this case, removing or dropping all missing values would be the best strategy. Salary is a critical metric for financial analysis. Imputing it risks skewing conclusions. Since only a small fraction (2.3%) is missing, I choose to drop the incomplete records.

- Age (211 missing values). Filling the missing values is the best strategy here. Age is often a feature for predictive modelling (e.g., turnover). Dropping 20% of my data is too costly. I’ll fill the missing values using the median age to avoid skewing the distribution with the mean.


In [1548]:
# Calculate the percentage of missing values in the 'salary' column
pource_salary= (df['salary'].isnull().sum()/ len(df)) * 100
print(f"percentage of missing values in 'salary': {pource_salary:.2f}%")

percentage of missing values in 'salary': 2.35%


In [1549]:
# Calculate the percentage of missing values in the 'age' column
pource_age= (df['age'].isnull().sum()/ len(df)) * 100
print(f"percentage of missing values in 'age': {pource_age:.2f}%")

percentage of missing values in 'age': 20.69%


In [1550]:
# 1. Removal: Drop rows missing the critical 'salary' data
df.dropna(subset=['salary'], inplace=True)

# 2. Imputation: Fill missing 'age' with the median
median_age = df['age'].median()
df['age'].fillna(median_age, inplace=True)

Next step:

By addressing the structural and missing data issues here, the subsequent steps can focus entirely on value standardisation, such as the messy 36 unique values in department_region—which we tackle in the next phase

## Step 4 — Value Standardization: Making Data Consistent

My DataFrame now has the right structure, but the values inside are still dirty. This step is about consistency. If “IT,” “i.t,” and “Info. Tech” all mean the same department, I need to force them into a single, clean value (“IT”). This prevents errors in grouping, filtering, and any statistical analysis based on categories.

1. Cleaning Corrupted String Data (The Phone Number Fix)


Remember the corrupted phone column from Step 2? It's currently a mess of negative scientific notation numbers that we converted to strings in Step 3. Now, it's time to extract the actual digits.

So, I’ll be removing every non-digit character (dashes, parentheses, dots, etc.) and converting the result into a clean, unified format. Regular expressions (.str.replace()) are perfect for this. I use \D to match any non-digit character and replace it with an empty string

In [1551]:
# The phone column is currently a string like '-9.994973e+09'
# We use regex to remove everything that isn't a digit
df['phone'] = df['phone'].str.replace(r'\D', '', regex=True)
# We can also truncate or format the resulting string if needed
# For example, keeping only the last 10 digits:
df['phone'] = df['phone'].str.slice(-10)
print(df['phone'])

employee_id
EMP1000    1651623197
EMP1001    1898471390
EMP1002    5596363211
EMP1003    3476490784
EMP1004    1586734256
              ...    
EMP2014    2470739200
EMP2016    2508261122
EMP2017    1261632487
EMP2018    8995729892
EMP2019    7629745492
Name: phone, Length: 996, dtype: object


2. Separating and Standardizing Categorical Data (Fixing the 36 Regions)

In [1552]:
df["department_region"].nunique()

36

My **df.nunique()** check revealed 36 unique values in the department_region column. When I reviewed all the unique values in the column, the output revealed that they are all neatly structured as **department-region** (e.g., devops-california, finance-texas, cloud tech-new york).

I guess one way to solve this is to split this single column into two dedicated columns. I'll split the column at the hyphen (-) and assign the parts to new columns: department and region.

In [1553]:
# 1. Split the combined column into two new, clean columns
df[['department', 'region']] = df['department_region'].str.split('-', expand=True)
# Next, I’ll drop the department_region column since it’s pretty much useless now
# 2. Drop the redundant combined column
df.drop('department_region', axis=1, inplace=True)
#Let’s review our new columns
print(df[['department', 'region']])

             department      region
employee_id                        
EMP1000          DevOps  California
EMP1001         Finance       Texas
EMP1002           Admin      Nevada
EMP1003           Admin      Nevada
EMP1004      Cloud Tech     Florida
...                 ...         ...
EMP2014         Finance      Nevada
EMP2016      Cloud Tech       Texas
EMP2017         Finance    New York
EMP2018              HR     Florida
EMP2019          DevOps    Illinois

[996 rows x 2 columns]


checking the unique values of the cleaned DataFrame


In [1554]:
print("Value unique in department column:", df['department'].nunique())
print("Value unique in region column:", df['region'].nunique())

Value unique in department column: 6
Value unique in region column: 6


checking the first 5 rows of the cleaned DataFrame


In [1555]:
df.head(5)

,first_name,last_name,age,status,join_date,salary,email,phone,performance_score,remote_work,department,region
employee_id,,,,,,,,,,,,
EMP1000,Bob,Davis,25.0,Active,4/2/2021,59767.65,bob.davis@example.com,1651623197,Average,True,DevOps,California
EMP1001,Bob,Brown,30.0,Active,7/10/2020,65304.66,bob.brown@example.com,1898471390,Excellent,True,Finance,Texas
EMP1002,Alice,Jones,30.0,Pending,12/7/2023,88145.90,alice.jones@example.com,5596363211,Good,True,Admin,Nevada
EMP1003,Eva,Davis,25.0,Inactive,11/27/2021,69450.99,eva.davis@example.com,3476490784,Good,True,Admin,Nevada
EMP1004,Frank,Williams,25.0,Active,1/5/2022,109324.61,frank.williams@example.com,1586734256,Poor,False,Cloud Tech,Florida


After splitting, the new **department** column has only 6 unique values (e.g., 'devops', 'finance', 'admin', etc.). This is great news. The values are already standardised and ready for analysis! I guess we could always map all similar departments to one single category. But I’m gonna skip that. I don’t want to get too advanced in this article.

3. Converting Date Columns (The Join_Date Fix)


The Join_Date column is usually read in as a string (object) type, which makes time-series analysis impossible. This means we have to convert it to a proper Pandas datetime object.

pd.to_datetime() is the core function. I often use errors='coerce' as a safety net; if Pandas can't parse a date, it converts that value to NaT (Not a Time), which is a clean null value, preventing the whole operation from crashing.

In [1556]:
df["join_date"].head()

employee_id
EMP1000      4/2/2021
EMP1001     7/10/2020
EMP1002     12/7/2023
EMP1003    11/27/2021
EMP1004      1/5/2022
Name: join_date, dtype: object

In [1557]:
# Convert the join_date column to datetime objects
df['join_date'] = pd.to_datetime(df['join_date'], errors='coerce')

check the data type of the **join_date** column

In [1558]:
df["join_date"].head()

employee_id
EMP1000   2021-04-02
EMP1001   2020-07-10
EMP1002   2023-12-07
EMP1003   2021-11-27
EMP1004   2022-01-05
Name: join_date, dtype: datetime64[ns]

note:
- The conversion of dates enables powerful time-series analysis, like calculating average employee tenure or identifying turnover rates by year.

- After this step, every value in the dataset is clean, uniform, and correctly formatted. The categorical columns (like department and region) are ready for grouping and visualisation, and the numerical columns (like salary and age) are ready for statistical modeling. The dataset is officially ready for

### Step 5 — Final Quality Check and Export



Before closing the notebook, I always perform one last audit to ensure everything is perfect, and then I export the data so I can perform analysis on it later.

The Final Data Quality Check
This is quick. I re-run the two most critical inspection methods to confirm that all my cleaning commands actually worked:

df.info(): I confirm there are no more missing values in the critical columns (age, salary) and that the data types are correct (phone is a string, join_date is datetime).

df.describe(): I ensure the statistical summary shows plausible numbers. The Phone column should now be absent from this output (since it's a string), and Age and Salary should have logical minimum and maximum values.

If these checks pass, I know the data is reliable.

In [1559]:
df.info()


<class 'pandas.core.frame.DataFrame'>
Index: 996 entries, EMP1000 to EMP2019
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   first_name         996 non-null    object        
 1   last_name          996 non-null    object        
 2   age                996 non-null    float64       
 3   status             996 non-null    object        
 4   join_date          996 non-null    datetime64[ns]
 5   salary             996 non-null    float64       
 6   email              996 non-null    object        
 7   phone              996 non-null    object        
 8   performance_score  996 non-null    object        
 9   remote_work        996 non-null    bool          
 10  department         996 non-null    object        
 11  region             996 non-null    object        
dtypes: bool(1), datetime64[ns](1), float64(2), object(8)
memory usage: 94.3+ KB


In [1560]:
df.describe()

,age,join_date,salary
count,996.000000,996,996.000000
mean,31.952811,2022-07-22 06:28:54.939759104,85155.056396
min,25.000000,2020-01-01 00:00:00,50047.320000
25%,30.000000,2021-03-29 18:00:00,68392.487500
50%,30.000000,2022-07-31 12:00:00,85547.870000
75%,35.000000,2023-11-03 06:00:00,100974.027500
max,40.000000,2024-12-29 00:00:00,119971.650000
std,5.124263,NaN,19873.727918


In [1561]:
df.head()

,first_name,last_name,age,status,join_date,salary,email,phone,performance_score,remote_work,department,region
employee_id,,,,,,,,,,,,
EMP1000,Bob,Davis,25.0,Active,2021-04-02,59767.65,bob.davis@example.com,1651623197,Average,True,DevOps,California
EMP1001,Bob,Brown,30.0,Active,2020-07-10,65304.66,bob.brown@example.com,1898471390,Excellent,True,Finance,Texas
EMP1002,Alice,Jones,30.0,Pending,2023-12-07,88145.90,alice.jones@example.com,5596363211,Good,True,Admin,Nevada
EMP1003,Eva,Davis,25.0,Inactive,2021-11-27,69450.99,eva.davis@example.com,3476490784,Good,True,Admin,Nevada
EMP1004,Frank,Williams,25.0,Active,2022-01-05,109324.61,frank.williams@example.com,1586734256,Poor,False,Cloud Tech,Florida


1. the end: Exporting the Clean Dataset


The final step is to save this cleaned version of the data. I usually save it as a new CSV file to keep the original messy file intact for reference. I use index=False here if I don't want the employee_id (which is now the index) to be saved as a separate column, or index=True if I want to save the index as the first column in the new CSV.

In [1562]:
# Exporting the clean DataFrame to a new CSV file
# We use index=True to keep our primary key (employee_id) in the exported file
df.to_csv('cleaned_employee_data.csv', index=True)

- By exporting with a clear, new filename (e.g., _clean.csv), you officially mark the end of the cleaning phase and provide a clean slate for the next phase of the project.

### Conclusion


Honestly, I used to feel overwhelmed by a messy dataset. The missing values, the weird data types, the cryptic columns — it felt like facing the blank page syndrome.

But this structured, repeatable workflow changed everything. By focusing on Load, Inspect, Clean, Review, and Export, we established order instantly: standardizing column names, making the employee_id the index, and using smart strategies for imputation and splitting messy columns.

Now, I can jump straight into the fun analysis part without constantly second-guessing my results. If you struggle with the initial data cleaning step, try out this workflow. I’d love to hear how it goes. If you want to play around with the dataset, you can download it here.
 

The project end date is 01/11/2026 17:24🐍✅✔️🌅
